In [ ]:
# this is for RIPPER
%pip install wittgenstein

In [ ]:
import os
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 21
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = ["accuracy", "precision", "recall", "f1", "roc_auc"]
COST_FN, COST_FP = 100, 10          # C(+,-), C(-,+)

DATA_PATH = "data/adult.data"

# the raw file has no header so we name the columns ourselves order comes from adult.names
COLUMNS = ["age", "workclass", "fnlwgt", "education", "education-num",
           "marital-status", "occupation", "relationship", "race", "sex",
           "capital-gain", "capital-loss", "hours-per-week", "native-country",
           "income"]
NUMERIC_COLS = ["age", "fnlwgt", "education-num", "capital-gain",
                "capital-loss", "hours-per-week"]
CATEGORICAL_COLS = ["workclass", "education", "marital-status", "occupation",
                    "relationship", "race", "sex", "native-country"]

# we keep all 14 attributes to match the spec, so nothing is dropped for now
DROP_COLS = []

Task 1 functions: load_data and make_preprocessor 

In [ ]:
def load_data(path=DATA_PATH) -> tuple[pd.DataFrame, pd.Series]:
    """Read the raw file and apply STATELESS cleaning only.
    - assign COLUMNS (file has no header)
    - strip leading whitespace from values
    - replace '?' with NaN
    - encode target: '>50K' -> 1, '<=50K' -> 0
    - drop columns listed in DROP_COLS
    Returns X (features) and y (0/1 target)."""

    df = pd.read_csv(path, names=COLUMNS, header=None,
                     skipinitialspace=True, na_values="?")

    # we encode the target as 1 for >50K and 0 for <=50K
    y = (df["income"].str.rstrip(".") == ">50K").astype(int)

    # we split off the features by dropping the target and any columns we chose to drop
    X = df.drop(columns=["income"] + DROP_COLS)

    return X, y


def make_preprocessor() -> ColumnTransformer:
    """Return an UNFITTED Task 1 ColumnTransformer.
    Never call .fit() here. It is fitted inside each CV fold."""

    numeric_features = [c for c in NUMERIC_COLS if c not in DROP_COLS]
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")), 
            ("scaler", StandardScaler())]
    )

    categorical_features = [c for c in CATEGORICAL_COLS if c not in DROP_COLS]
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

Task 2 evaluation harness

In [ ]:
def evaluate(pipeline, X, y) -> dict:

    fold_scores = {m: [] for m in SCORING}

    # one out-of-fold probability per row, filled in as each fold is scored
    oof_proba = np.zeros(len(y))

    # this loop is where cross-validation happens: 5 folds, each one a train/test split
    for train_idx, test_idx in CV.split(X, y):

        # we clone so every fold starts from a fresh, unfitted copy of the pipeline
        model = clone(pipeline)

        model.fit(X.iloc[train_idx], y.iloc[train_idx])

        # we score on the held-out rows the model never saw
        y_true = y.iloc[test_idx]
        proba = model.predict_proba(X.iloc[test_idx])[:, 1]
        pred = (proba > 0.5).astype(int)   # default 0.5 threshold; Task 4 moves this

        fold_scores["accuracy"].append(accuracy_score(y_true, pred))
        fold_scores["precision"].append(precision_score(y_true, pred, zero_division=0))
        fold_scores["recall"].append(recall_score(y_true, pred, zero_division=0))
        fold_scores["f1"].append(f1_score(y_true, pred, zero_division=0))
        fold_scores["roc_auc"].append(roc_auc_score(y_true, proba))

        oof_proba[test_idx] = proba

    metrics = {m: (np.mean(v), np.std(v)) for m, v in fold_scores.items()}
    return {"metrics": metrics, "oof_proba": oof_proba}

Task 4 cost function: cost_sweep 

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
def cost_sweep(y, oof_proba, thresholds=np.linspace(0, 1, 101)) -> pd.DataFrame:
    """For each threshold t, predict + when P(+|x) > t and compute
    total cost = FN * COST_FN + FP * COST_FP.
    Returns a DataFrame with columns: threshold, FN, FP, total_cost.
    Theoretical optimum: COST_FP / (COST_FP + COST_FN) = 10/110 ≈ 0.091."""
    
    cost_results = []
    
    for threshold in thresholds:
        predictions = (oof_proba >= threshold).astype(int)
        
        tn, fp, fn, tp = confusion_matrix(y, predictions, labels=[0, 1]).ravel()
        
        total_cost = (100 * fn) + (10 * fp) 
        
        cost_results.append({
        "threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "total_cost": total_cost,
        "cost_per_person": total_cost / len(y),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0)
        })
        
    cost_dataframe = pd.DataFrame(cost_results)
    
    return cost_dataframe

Rule-based model wrapper

In [ ]:
import numpy as np
import pandas as pd
import wittgenstein as lw
from scipy import sparse
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.utils.multiclass import unique_labels
from sklearn.utils.validation import check_is_fitted



# Ripper wrapper for skicit
class RipperClassifier(ClassifierMixin, BaseEstimator):


    def __init__(self, k=2, prune_size=0.33, dl_allowance=64, n_discretize_bins=10,
                 max_rules=None, max_rule_conds=None, random_state=None):
        self.k = k
        self.prune_size = prune_size
        self.dl_allowance = dl_allowance
        self.n_discretize_bins = n_discretize_bins
        self.max_rules = max_rules
        self.max_rule_conds = max_rule_conds
        self.random_state = random_state

    def _to_frame(self, X):
        if isinstance(X, pd.DataFrame):
            return X.reset_index(drop=True)
        if sparse.issparse(X):
            X = X.toarray()
        X = np.asarray(X)
        return pd.DataFrame(X, columns=[f"x{i}" for i in range(X.shape[1])])

    def fit(self, X, y):
        X = self._to_frame(X)
        y = np.asarray(y)
        if len(X) != len(y):
            raise ValueError("X and y have different numbers of rows")

        # we store the classes seen during fit predict_proba columns follow this order
        self.classes_ = unique_labels(y)
        if not np.array_equal(self.classes_, [0, 1]):
            raise ValueError("RipperClassifier expects a binary 0/1 target")
        self.n_features_in_ = X.shape[1]

        # we build a fresh RIPPER every fit so a refit never reuses old rules
        self.model_ = lw.RIPPER(
            k=self.k,
            prune_size=self.prune_size,
            dl_allowance=self.dl_allowance,
            n_discretize_bins=self.n_discretize_bins,
            max_rules=self.max_rules,
            max_rule_conds=self.max_rule_conds,
            random_state=self.random_state,
        )
        self.model_.fit(X, y, pos_class=1)
        return self

    def predict_proba(self, X):
        check_is_fitted(self)
        return np.asarray(self.model_.predict_proba(self._to_frame(X)), dtype=float)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

    @property
    def ruleset_(self):
        check_is_fitted(self)
        return self.model_.ruleset_


# rule preproccessor 
def make_rule_preprocessor() -> ColumnTransformer:
    
    num_cols = [c for c in NUMERIC_COLS if c not in DROP_COLS]
    cat_cols = [c for c in CATEGORICAL_COLS if c not in DROP_COLS]
    return ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), num_cols),
            ("cat", SimpleImputer(strategy="most_frequent"), cat_cols)],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")


def make_ripper_pipeline() -> Pipeline:
    # Unfitted pipeline: rule preprocessor + RIPPER, using our fixed step names
    return Pipeline(steps=[
        ("prep", make_rule_preprocessor()),
        ("clf", RipperClassifier(random_state=RANDOM_STATE))])

Task 1

In [ ]:
#load the data and check it matches the spec
X, y = load_data()

print("Rows:", len(X))                      # expect 32561
print("Share >50K:", round(y.mean(), 3))    # expect ~0.241

In [ ]:
# checking to see what has missing values
X.isna().sum()

In [ ]:
# inspecting the shape 
print(make_preprocessor().fit_transform(X).shape)

Task 2

In [ ]:
# we collect every model's results here
results = {}

# small helper so every model prints the same way mean ± std for each metric
def show(name, res):
    print(name)
    for m, (mean, std) in res["metrics"].items():
        print(f"  {m:<10} {mean:.3f} ± {std:.3f}")

In [ ]:
#baseline pipeline 
dummy_pipe = Pipeline(
    steps=[
        ("prep", make_preprocessor()),
        ("clf", DummyClassifier(strategy="most_frequent"))])

results["Dummy (majority)"] = evaluate(dummy_pipe, X, y)

show("Dummy (majority)", results["Dummy (majority)"])

In [ ]:
#default tree
dt_default = Pipeline(
    steps=[
        ("prep", make_preprocessor()),
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))])

results["Decision Tree (default)"] = evaluate(dt_default, X, y)

show("Decision Tree (default)", results["Decision Tree (default)"])

Task 3 

In [ ]:
dt_pipe = Pipeline(
    steps=[
        ("prep", make_preprocessor()),
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))])

param_grid = {
    "clf__max_depth": list(range(3, 21)),                  
    "clf__min_samples_split": [2, 10, 50, 100, 200],       
    "clf__criterion": ["gini", "entropy"],
}

print("Candidates:", 18 * 5 * 2, "x 5 folds =", 18 * 5 * 2 * 5, "fits")

In [ ]:
#winner by F1 
dt_search = GridSearchCV(
    dt_pipe,
    param_grid=param_grid,
    scoring=SCORING,
    refit="f1",
    cv=CV,
    n_jobs=-1,
    return_train_score=True,
)
dt_search.fit(X, y)  

print("Best parameters:", dt_search.best_params_)
print("Best mean CV F1:", round(dt_search.best_score_, 4))

In [ ]:
#top 10 candidates by F1, with the other metrics alongside
cv_table = pd.DataFrame(dt_search.cv_results_)
cols = ["param_clf__criterion", "param_clf__max_depth", "param_clf__min_samples_split",
        "mean_test_f1", "std_test_f1", "mean_train_f1", "mean_test_roc_auc", "mean_test_accuracy"]
cv_table.sort_values("rank_test_f1")[cols].head(10).round(4)

In [ ]:
best_split = dt_search.best_params_["clf__min_samples_split"]
fig, ax = plt.subplots(figsize=(9, 5))

for criterion, color in [("gini", "tab:blue"), ("entropy", "tab:orange")]:
    rows = cv_table[(cv_table["param_clf__criterion"] == criterion) &
                    (cv_table["param_clf__min_samples_split"] == best_split)].sort_values("param_clf__max_depth")
    depth = rows["param_clf__max_depth"].astype(int)
    #test folds with a ±1 std band, training folds dashed
    ax.plot(depth, rows["mean_test_f1"], "-", color=color, label=f"{criterion} (test)")
    ax.fill_between(depth, rows["mean_test_f1"] - rows["std_test_f1"],
                    rows["mean_test_f1"] + rows["std_test_f1"], color=color, alpha=0.15)
    ax.plot(depth, rows["mean_train_f1"], "--", color=color, alpha=0.7, label=f"{criterion} (train)")

best_depth = dt_search.best_params_["clf__max_depth"]
ax.axvline(best_depth, linestyle=":", color="gray")
ax.annotate(f"best: depth {best_depth}", (best_depth + 0.2, dt_search.best_score_ - 0.03))
ax.set_xticks(range(3, 21))
ax.set_xlabel("max_depth")
ax.set_ylabel("F1 (positive class >50K)")
ax.set_title(f"Decision tree F1 vs. depth (min_samples_split={best_split})")
ax.legend()
plt.show()

In [ ]:
#evaluate() clones the pipeline, so this re-fits the best settings fold by fold
dt_best = dt_search.best_estimator_
results["Decision Tree (tuned)"] = evaluate(dt_best, X, y)
show("Decision Tree (tuned)", results["Decision Tree (tuned)"])

task 3 k-NN tuning


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
# k-NN pipeline and grid search
knn_pipeline = Pipeline(
    steps=[
        ("prep", make_preprocessor()),
        ("model", KNeighborsClassifier())])

knn_grid = {
    "model__n_neighbors": [1, 3, 7, 15, 31],
    "model__weights": ['uniform', 'distance'],
    "model__metric": ["euclidean", "manhattan"]
}

knn_search = GridSearchCV(
    knn_pipeline,
    param_grid=knn_grid,
    scoring=SCORING,
    refit="f1",
    cv=CV,
    n_jobs=-1,
    return_train_score=True
)



In [ ]:
knn_search.fit(X, y)

In [ ]:
print("Best parameters:", knn_search.best_params_)
print("Best mean CV F1:", round(knn_search.best_score_, 4))

In [ ]:
#top 10 candidates by F1, with the other metrics alongside
cv_table2 = pd.DataFrame(knn_search.cv_results_)
cols2 = ["param_model__metric", "param_model__n_neighbors", "param_model__weights",
        "mean_test_f1", "std_test_f1", "mean_train_f1", "mean_test_roc_auc", "mean_test_accuracy"]
cv_table2.sort_values("rank_test_f1")[cols2].head(10).round(4)

In [ ]:
# score the tuned k-NN through the same harness
best_split = knn_search.best_params_["model__weights"]
fig, ax = plt.subplots(figsize=(9, 5))

for criterion, color in [("euclidean", "tab:blue"), ("manhattan", "tab:orange")]:
    rows = cv_table2[(cv_table2["param_model__metric"] == criterion) &
                    (cv_table2["param_model__weights"] == best_split)].sort_values("param_model__n_neighbors")
    neighbors = rows["param_model__n_neighbors"].astype(int)
    #test folds with a ±1 std band, training folds dashed
    ax.plot(neighbors, rows["mean_test_f1"], "-", color=color, label=f"{criterion} (test)")
    ax.fill_between(neighbors, rows["mean_test_f1"] - rows["std_test_f1"],
                    rows["mean_test_f1"] + rows["std_test_f1"], color=color, alpha=0.15)
    ax.plot(neighbors, rows["mean_train_f1"], "--", color=color, alpha=0.7, label=f"{criterion} (train)")

best_neighbors = knn_search.best_params_["model__n_neighbors"]
ax.axvline(best_neighbors, linestyle=":", color="gray")
ax.annotate(f"best: neighbors {best_neighbors}", (best_neighbors + 0.2, knn_search.best_score_ - 0.03))
ax.set_xticks([1, 3, 7, 15, 31])
ax.set_xlabel("n_neighbors")
ax.set_ylabel("F1 (positive class >50K)")
ax.set_title(f"KNN search F1 vs. neighbors (min_samples_split={best_split})")
ax.legend()
plt.show()

In [ ]:
knn_best = knn_search.best_estimator_
results["KNN Search (tuned)"] = evaluate(knn_best, X, y)
show("KNN Search (tuned)", results["KNN Search (tuned)"])

RIPPER 

In [ ]:
ripper_pipe = make_ripper_pipeline()

results["RIPPER"] = evaluate(ripper_pipe, X, y)
show("RIPPER", results["RIPPER"])

In [ ]:
ripper_full = make_ripper_pipeline().fit(X, y)
rules = ripper_full.named_steps["clf"].ruleset_

print("Number of rules:", len(rules))
print("Each rule predicts >50K; anything no rule covers is predicted <=50K\n")
for rule in rules[:10]:
    print(" ", rule)

task 2 benchmark bar plot

In [ ]:
#plot_metrics(results)

def plot_metrics(results):
    best_row = results.loc[results["total_cost"].idxmin()]
    print("Optimal theshold on OOF predictsions:")
    print(best_row.to_string())
    
    plt.figure(figsize=(9, 5))
    plt.plot(
        results["threshold"],
        results["total_cost"]
    )
    plt.axvline(
        best_row["threshold"],
        linestyle="--",
        label=f'Best threshold = {best_row["threshold"]:.2f}'
    )
    plt.xlabel("Classification threshold")
    plt.ylabel("Total misclassification cost ($)")
    plt.title("Threshold Selection by Total Cost")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    

In [ ]:
#one row per model, mean ± std from the same 5 folds
summary = pd.DataFrame({
    name: {m: f"{mean:.3f} ± {std:.3f}" for m, (mean, std) in res["metrics"].items()}
    for name, res in results.items()
}).T
summary

In [ ]:
#we save the results (including out-of-fold probabilities) so Task 4 and the final notebook can reuse them
os.makedirs("artifacts", exist_ok=True)
joblib.dump({"results": results, "dt_best_params": dt_search.best_params_},
            "artifacts/a_results.joblib")
print("saved artifacts/a_results.joblib")

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [ ]:
# SMOTE pipeline
smote_knn_pipeline = ImbPipeline([
    ("preprocessor", make_preprocessor()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", KNeighborsClassifier())
])

tree_grid = {
    "model__max_depth": list(range(3, 21)),
    "model__min_samples_split": [2, 5, 10, 20],
    "model__criterion": ["gini", "entropy"]
}
smote_tree_pipeline = ImbPipeline([
    ("preprocessor", make_preprocessor()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
])

smote_knn_search = GridSearchCV(
    smote_knn_pipeline,
    param_grid=knn_grid,
    scoring=SCORING,
    refit="f1",
    cv=CV,
    n_jobs=-1,
    return_train_score=True,
)

smote_tree_search = GridSearchCV(
    smote_tree_pipeline,
    param_grid=tree_grid,
    scoring=SCORING,
    refit="f1",
    cv=CV,
    n_jobs=-1,
    return_train_score=True,
)

In [ ]:
smote_tree_search.fit(X, y)


In [ ]:
smote_knn_search.fit(X, y)

In [ ]:
from sklearn.model_selection import cross_val_predict

In [ ]:
#cost sweep and cost-vs-threshold plot
best_search = max([smote_knn_search, smote_tree_search],
                  key=lambda search: search.best_score_)

best_model = best_search.best_estimator_

print("Selected model:", type(best_model.named_steps["model"]).__name__)
print("Best CV F1:", best_search.best_score_)

# Obtain out-of-fold probabilities for threshold selection.
# Each example's probability comes from a model that did not train on it.
oof_probabilities = cross_val_predict(
    best_model,
    X,
    y,
    cv=CV,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

results = cost_sweep(y, oof_probabilities)

In [ ]:
plot_metrics(results)

In [ ]:
# final debugged pipeline: the AI's code with only the data loading fixed
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# -------------------------------------------------
# Load dataset
# -------------------------------------------------
# FIX for bug 1 and bug 2: load_data() names the columns, turns ? into NaN, and encodes >50K as 1

# COLUMNS = ["age", "workclass", "fnlwgt", "education", "education-num",
#            "marital-status", "occupation", "relationship", "race", "sex",
#            "capital-gain", "capital-loss", "hours-per-week", "native-country",
#            "income"]

# def load_data(path="data/adult.data") -> tuple[pd.DataFrame, pd.Series]:

#     df = pd.read_csv(path, names=COLUMNS, header=None,
#                      skipinitialspace=True, na_values="?")

#     # we encode the target as 1 for >50K and 0 for <=50K
#     y = (df["income"].str.rstrip(".") == ">50K").astype(int)

#     # we split off the features by dropping the target and any columns we chose to drop
#     X = df.drop(columns=["income"])

#     return X, y

X, y = load_data()

# -------------------------------------------------
# Identify feature types
# -------------------------------------------------
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "category"]).columns

# -------------------------------------------------
# Preprocessing
# -------------------------------------------------
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# -------------------------------------------------
# Train/Test Split
# -------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------------------------------------
# Pipeline
# -------------------------------------------------
pipeline = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

# -------------------------------------------------
# Hyperparameter Grid
# -------------------------------------------------
param_grid = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [None, 5, 10, 20],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4]
}

# -------------------------------------------------
# Grid Search
# -------------------------------------------------
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

# Train
grid_search.fit(X_train, y_train)

# -------------------------------------------------
# Results
# -------------------------------------------------
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(grid_search.best_score_)

# -------------------------------------------------
# Evaluate
# -------------------------------------------------
y_pred = grid_search.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))